# Nisjemodell — raster → vektor (03a)

Bygger det nasjonale NiN-rasteret fra Lys + Salinitet + Substrat, vektoriserer og
generaliserer det (GRASS), og skriver `…_generalized.gpkg`.

Vektor-overleggene (tørrfall, observasjons-substrat, land, attributter) ligger i
**`03b_nisjemodell_vektor.ipynb`**, som leser gpkg-en herfra. Delingen gjør at man
kan iterere på overleggene uten å kjøre den dyre raster-/GRASS-delen på nytt.

Pipeline (03a): Lys + Salinitet + Substrat → vindus-vis rasterklassifisering
(3-bit nøkkel) → vektorisering + generalisering (GRASS).

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from rasterio.enums import Resampling

import mnk
import mnk.vectorize
import mnk.rules

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Kilder og stier

- **Lys / referansegrid**: nasjonalt fotisk-sone-raster (50 m, EPSG:25833, 1=Eufotisk, 0=Afotisk, -1=nodata).
- **Brakkvann**: `hi/Norfjords_160_surf_salt_mean_UTM33_brakkvann.tif` (EPSG:32633, 1=brakk, 0=salt, 255=nodata).
- **Substrat**: nasjonal `nisjedata-substrat_norge` geoparquet (`BunnType`).
- **Tørrfall**: `mnk.sources.sea_map_torrfall()` (alle fylker) — brukes i vektor-steget.
- **Land**: nasjonal `Basisdata_Landareal` geoparquet.

In [ ]:
REPO = Path.cwd()

photic_path = "gs://niva-geodata/MarintNaturKart/results/nisjedata-fotisk-sone-kd2020-2025_norge_2026_25833_padded.tif"
brakkvann_path = "gs://niva-geodata/MarintNaturKart/results/Norfjords_160_surf_salt_mean_UTM33_brakkvann_50m_filled.tif"

# 0=soft, 1=mixed, 2=hard, 255=nodata
substrat_raster_path = (
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833_2class_padded.tif"
)

nin_raster_path = REPO / "nisjedata-modell_norge_2026_25833.tif"
nin_vec_path = REPO / "nisjedata-modell_norge_2026_25833_generalized.gpkg"  # -> 03b leser denne

## NiN-koding

Vi koder tre binære dimensjoner (Lys, Salinitet, Substrat) til en 3-bit nøkkel og
antar foreløpig **Utenfor** tørrfall overalt. Tørrfall legges på i vektor-steget.

```
key = lys*4 + salinitet*2 + substrat
```

In [ ]:
# NiN integer (1-12) -> (nin_kode_fkid, NiN_name), derived from mnk.rules
# (single source of truth: mnk/rules.py). LUT3 maps the 3 binary axes (tørrfall
# = outside) onto these integer codes for the raster step.
NIN_INT_MAP = {
    1: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "innenfor"),   # NA-MC04
    2: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "innenfor"),   # NA-MA04
    3: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "innenfor"),   # NA-MC01
    4: mnk.rules.classify_nisje("eufotisk", "salt",  "fast", "innenfor"),   # NA-MA01
    5: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "utenfor"),    # NA-MC04
    6: mnk.rules.classify_nisje("afotisk",  "brakk", "løst", "utenfor"),    # NA-MC04
    7: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "utenfor"),    # NA-MA05
    8: mnk.rules.classify_nisje("afotisk",  "salt",  "løst", "utenfor"),    # NA-MA06
    9: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    10: mnk.rules.classify_nisje("afotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    11: mnk.rules.classify_nisje("eufotisk", "salt", "fast", "utenfor"),    # NA-MA02
    12: mnk.rules.classify_nisje("afotisk", "salt", "fast", "utenfor"),     # NA-MA03
}

# 3-bit LUT (tørrfall = outside):  key = lys*4 + salinitet*2 + substrat
#   0: Aphotic  Salt     Soft -> 8     1: Aphotic  Salt     Hard -> 12
#   2: Aphotic  Brackish Soft -> 6     3: Aphotic  Brackish Hard -> 10
#   4: Euphotic Salt     Soft -> 7     5: Euphotic Salt     Hard -> 11
#   6: Euphotic Brackish Soft -> 5     7: Euphotic Brackish Hard -> 9
LUT3 = np.array([8, 12, 6, 10, 7, 11, 5, 9], dtype=np.uint8)
print(LUT3)

## Vindus-vis rasterklassifisering

For hvert rad-stripe-vindu:
1. les Lys fra referansegridet,
2. reprojiser brakkvann (WarpedVRT) inn på gridet,
3. reprojiser modellert substrat-raster (WarpedVRT) inn på gridet
   (2=fastbunn → Fast, ellers Løst),
4. kod til NiN (Utenfor-antakelse) og skriv blokk til midlertidig GeoTIFF.

In [ ]:
BLOCK_ROWS = 2048

with rasterio.open(photic_path) as photic:
    transform = photic.transform
    width, height = photic.width, photic.height
    photic_nodata = photic.nodata if photic.nodata is not None else -1

    profile = dict(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=CRS,
        transform=transform,
        nodata=NODATA,
        compress="deflate",
        tiled=True,
        blockxsize=512,
        blockysize=512,
    )

    brak_src = rasterio.open(brakkvann_path)
    brak_vrt = WarpedVRT(
        brak_src,
        crs=CRS,
        transform=transform,
        width=width,
        height=height,
        resampling=Resampling.nearest,
        src_nodata=brak_src.nodata,
        nodata=255,
    )

    sub_src = rasterio.open(substrat_raster_path)
    sub_vrt = WarpedVRT(
        sub_src,
        crs=CRS,
        transform=transform,
        width=width,
        height=height,
        resampling=Resampling.nearest,
        src_nodata=sub_src.nodata,
        nodata=255,
    )

    counts = np.zeros(13, dtype=np.int64)
    with rasterio.open(nin_raster_path, "w", **profile) as dst:
        for row0 in range(0, height, BLOCK_ROWS):
            nrows = min(BLOCK_ROWS, height - row0)
            win = Window(0, row0, width, nrows)
            shp = (nrows, width)

            lys_raw = photic.read(1, window=win)
            lys_valid = lys_raw != photic_nodata
            lys = (lys_raw == 1).astype(np.uint8)  # 1=euphotic, 0=aphotic

            # Brackish only where VRT==1; coverage gaps count as salt
            brak = brak_vrt.read(1, window=win)
            salinitet = (brak == 1).astype(np.uint8)

            # 2=hard -> 1, 0/1 -> soft (0)
            sub_raw = sub_vrt.read(1, window=win)
            sub_valid = sub_raw != 255
            substrat = (sub_raw == 2).astype(np.uint8)

            valid = lys_valid & sub_valid
            nin = np.full(shp, NODATA, dtype=np.uint8)
            key = (lys[valid] * 4 + salinitet[valid] * 2 + substrat[valid]).astype(np.uint8)
            nin[valid] = LUT3[key]
            dst.write(nin, 1, window=win)

            for v in np.unique(nin[nin != NODATA]):
                counts[v] += int(np.sum(nin == v))
            print(f"  rows {row0:>6}-{row0+nrows:<6} valid={int(valid.sum()):>12,}", end="\r")

    brak_vrt.close()
    brak_src.close()
    sub_vrt.close()
    sub_src.close()

print("\nNiN-fordeling (før tørrfall/observasjon/land):")
for i, (code_, name) in NIN_INT_MAP.items():
    if counts[i]:
        print(f"  {i:2d} {code_} {name}: {counts[i]:,}")

## Vektoriser + generaliser (GRASS)

Vektoriserer NiN-rasteret og fjerner 50 m trappetrinn-artefaktene i samme, disk-streamede operasjon via GRASS `r.to.vect` → `v.generalize` (Douglas–Peucker, terskel 25 m). Topologi bevares slik at felles grenser mellom NiN-klasser ikke får glipper/overlapp. Minnegjerrig fordi GRASS strømmer mot disk.

In [ ]:
mnk.vectorize.with_grass(nin_raster_path, nin_vec_path)

## Neste steg

Kjør **`03b_nisjemodell_vektor.ipynb`** — den leser `nin_vec_path`
(`…_generalized.gpkg`) og legger på tørrfall, observasjons-substrat, land og
Naturkart-attributter.